# 🎄 Santa's Tree Packing Challenge 2025 🎅

## Shake & Optimize: A Simple Yet Effective Approach

**Competition:** [Santa 2025 - Christmas Tree Packing](https://www.kaggle.com/competitions/santa-2025)

**Approach:** The idea is simple - shake the solution in almost random directions and wait to see if the size decreases. This straightforward strategy leverages local search with random perturbations to escape local minima and find better packing configurations.

*Note: Code may be published after the competition ends.*

---

In [ ]:
source_file = "/kaggle/input/merging/submission.csv"

#source_file = "/kaggle/input/shake-shake-shake-bf2681/submission_fixed.csv"

In [ ]:
!cp /kaggle/input/shake-public/shake_public /kaggle/working/shake_public
!cp /kaggle/input/merging/submission.csv /kaggle/working/submission.csv
!chmod +x /kaggle/working/shake_public

In [ ]:
#!/kaggle/working/shake_public --input="$source_file" --output="submission.csv"

In [ ]:
import os, shutil, subprocess
import pandas as pd
from decimal import Decimal, getcontext
from shapely import affinity
from shapely.geometry import Polygon
from shapely.strtree import STRtree

# -------------------------
# Config
# -------------------------
getcontext().prec = 25
scale_factor = Decimal("1e18")

SHAKE_BIN = "/kaggle/working/shake_public"
assert os.path.exists(SHAKE_BIN), f"{SHAKE_BIN} not found. Did you cp+chmod it?"

class ChristmasTree:
    def __init__(self, center_x="0", center_y="0", angle="0"):
        self.center_x = Decimal(center_x)
        self.center_y = Decimal(center_y)
        self.angle = Decimal(angle)

        trunk_w = Decimal("0.15")
        trunk_h = Decimal("0.2")
        base_w  = Decimal("0.7")
        mid_w   = Decimal("0.4")
        top_w   = Decimal("0.25")
        tip_y   = Decimal("0.8")
        tier_1_y = Decimal("0.5")
        tier_2_y = Decimal("0.25")
        base_y  = Decimal("0.0")
        trunk_bottom_y = -trunk_h

        initial_polygon = Polygon([
            (Decimal("0.0") * scale_factor, tip_y * scale_factor),
            (top_w / Decimal("2") * scale_factor, tier_1_y * scale_factor),
            (top_w / Decimal("4") * scale_factor, tier_1_y * scale_factor),
            (mid_w / Decimal("2") * scale_factor, tier_2_y * scale_factor),
            (mid_w / Decimal("4") * scale_factor, tier_2_y * scale_factor),
            (base_w / Decimal("2") * scale_factor, base_y * scale_factor),
            (trunk_w / Decimal("2") * scale_factor, base_y * scale_factor),
            (trunk_w / Decimal("2") * scale_factor, trunk_bottom_y * scale_factor),
            (-(trunk_w / Decimal("2")) * scale_factor, trunk_bottom_y * scale_factor),
            (-(trunk_w / Decimal("2")) * scale_factor, base_y * scale_factor),
            (-(base_w / Decimal("2")) * scale_factor, base_y * scale_factor),
            (-(mid_w / Decimal("4")) * scale_factor, tier_2_y * scale_factor),
            (-(mid_w / Decimal("2")) * scale_factor, tier_2_y * scale_factor),
            (-(top_w / Decimal("4")) * scale_factor, tier_1_y * scale_factor),
            (-(top_w / Decimal("2")) * scale_factor, tier_1_y * scale_factor),
        ])

        rotated = affinity.rotate(initial_polygon, float(self.angle), origin=(0, 0))
        self.polygon = affinity.translate(
            rotated,
            xoff=float(self.center_x * scale_factor),
            yoff=float(self.center_y * scale_factor)
        )

def has_overlap(trees):
    if len(trees) <= 1:
        return False
    polys = [t.polygon for t in trees]
    idx = STRtree(polys)
    for i, p in enumerate(polys):
        hits = idx.query(p)
        for j in hits:
            if j == i:
                continue
            if p.intersects(polys[j]) and not p.touches(polys[j]):
                return True
    return False

def load_trees_for_n(n, df):
    group = df[df["id"].str.startswith(f"{n:03d}_")]
    trees = []
    for _, row in group.iterrows():
        x = str(row["x"]).lstrip("s")
        y = str(row["y"]).lstrip("s")
        deg = str(row["deg"]).lstrip("s")
        if x and y and deg:
            trees.append(ChristmasTree(x, y, deg))
    return trees

def fix_invalid_submission(new_csv_path, valid_csv_path, output_csv_path, max_n=200):
    df_new = pd.read_csv(new_csv_path)
    df_valid = pd.read_csv(valid_csv_path)
    replaced_n = []
    for n in range(1, max_n + 1):
        trees = load_trees_for_n(n, df_new)
        if trees and has_overlap(trees):
            prefix = f"{n:03d}_"
            df_new = df_new[~df_new["id"].str.startswith(prefix)]
            df_rep = df_valid[df_valid["id"].str.startswith(prefix)]
            df_new = pd.concat([df_new, df_rep], ignore_index=True)
            replaced_n.append(n)
    df_new = df_new.sort_values("id").reset_index(drop=True)
    df_new.to_csv(output_csv_path, index=False)
    return replaced_n

def run_one_shake(iter_idx: int, input_csv: str, valid_ref_csv: str, out_dir="shakes_chain"):
    os.makedirs(out_dir, exist_ok=True)

    # run shake_public
    cmd = [SHAKE_BIN, f'--input={input_csv}', '--output=submission.csv']
    subprocess.run(cmd, check=True)

    raw_csv = f"{out_dir}/submission_raw_{iter_idx:03d}.csv"
    shutil.copy("/kaggle/working/submission.csv", raw_csv)

    fixed_csv = f"{out_dir}/submission_fixed_{iter_idx:03d}.csv"
    replaced = fix_invalid_submission(raw_csv, valid_ref_csv, fixed_csv, max_n=200)
    return raw_csv, fixed_csv, replaced

# -------------------------
# Loop (chain mode)
# -------------------------
source_file = "/kaggle/input/merging/submission.csv"
current_input = source_file

for i in range(1, 101):
    raw_csv, fixed_csv, replaced = run_one_shake(i, current_input, source_file, out_dir="shakes_chain")
    print(f"[{i:03d}] replaced={len(replaced)} -> {fixed_csv}")
    current_input = fixed_csv

print("Final:", current_input)